### Imports

In [1]:
from pathlib import Path

import pandas as pd
import sweetviz as sv

### Paths

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "data"

NORMAL_CSV = DATA_PATH / "raw_cdr" / "cdr_6.8million_30kMSISDNS_v3.csv"
FRAUD_CSV = DATA_PATH / "raw_cdr" / "ali_sleiman_msisdns_cdr.csv"

NORMAL_PARQUET = DATA_PATH / "raw_cdr" / "normal.parquet"
FRAUD_PARQUET = DATA_PATH / "raw_cdr" / "fraud.parquet"

REPORTS_PATH = PROJECT_ROOT / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

### Convert CSV to Parquet

In [3]:
normal_df_csv = pd.read_csv(NORMAL_CSV)
fraud_df_csv = pd.read_csv(FRAUD_CSV)

normal_df_csv.to_parquet(NORMAL_PARQUET, index=False)
fraud_df_csv.to_parquet(FRAUD_PARQUET, index=False)

print("Parquet files created.")

Parquet files created.


### Load Parquet into dataframes

In [4]:
normal_df = pd.read_parquet(NORMAL_PARQUET)
fraud_df = pd.read_parquet(FRAUD_PARQUET)

print("Normal:", normal_df.shape)
print("Fraud :", fraud_df.shape)

Normal: (6849575, 8)
Fraud : (368754, 8)


---
### Partial Cleaning Function
- fixes column naming schema using snake_case <br>
- fixes column dtypes

In [5]:
def partial_clean(df):
    df = df.copy()

    # ====================================
    # Explicit column renaming
    # ====================================

    df = df.rename(columns={
        "MSISDN": "msisdn",
        "AnswerTime": "answer_time",
        "DurationInSeconds": "duration_in_seconds",
        "LAC": "lac",
        "Cell": "cell",
        "CallingNumber": "calling_number",
        "CalledNumber": "called_number",
        "RecordType": "record_type"
    })

    # ====================================
    # Fix dtypes
    # ====================================

    df["answer_time"] = pd.to_datetime(
        df["answer_time"],
        errors="coerce"
    )

    numeric_cols = [
        "msisdn",
        "duration_in_seconds",
        "lac",
        "cell",
        "calling_number",
        "called_number",
        "record_type"
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    return df

### Apply cleaning

In [6]:
normal_df = partial_clean(normal_df)
fraud_df = partial_clean(fraud_df)

normal_df.info()

fraud_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6849575 entries, 0 to 6849574
Data columns (total 8 columns):
 #   Column               Dtype         
---  ------               -----         
 0   msisdn               int64         
 1   answer_time          datetime64[ns]
 2   duration_in_seconds  int64         
 3   lac                  int64         
 4   cell                 int64         
 5   calling_number       float64       
 6   called_number        int64         
 7   record_type          int64         
dtypes: datetime64[ns](1), float64(1), int64(6)
memory usage: 418.1 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 368754 entries, 0 to 368753
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   msisdn               368754 non-null  int64         
 1   answer_time          368754 non-null  datetime64[ns]
 2   duration_in_seconds  368754 non-null  int64         
 3   la

---
### Comparison sweetviz report

In [11]:
comparison_report = sv.compare(
    [normal_df, "Normal Traffic"],
    [fraud_df, "Fraud Traffic"]
)

comparison_report.show_html(
    filepath=str(REPORTS_PATH / "sweetviz_comparison.html"),
    open_browser=False
)

                                             |          | [  0%]   00:00 -> (? left)

Report c:\Users\ali.farhat\Documents\simbox-project\reports\sweetviz_comparison.html was generated.


---
## EXTRA: daily profiles sweetviz reports

note: see notebook `05_profiling_and_feature_extraction.ipynb` for context.

### Paths

In [7]:
PROFILE_PATH = DATA_PATH / "profiles"

REPORT_PATH = PROJECT_ROOT / "reports"

REPORT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

### Look for daily profiles

In [8]:
profile_files = sorted(
    PROFILE_PATH.glob("day_*.parquet")
)

print(
    f"Found {len(profile_files)} profile files."
)

for file in profile_files:
    print(file.name)

Found 15 profile files.
day_01.parquet
day_02.parquet
day_03.parquet
day_04.parquet
day_05.parquet
day_06.parquet
day_07.parquet
day_08.parquet
day_09.parquet
day_10.parquet
day_11.parquet
day_12.parquet
day_13.parquet
day_14.parquet
day_15.parquet


### Generate sweetviz reports

In [11]:
for profile_file in profile_files:

    print(
        f"Generating report for "
        f"{profile_file.name}..."
    )

    df = pd.read_parquet(
        profile_file
    )

    report = sv.analyze(
        df,
        target_feat= "label",
        pairwise_analysis="on"
    )

    output_file = (
        REPORT_PATH
        /
        f"{profile_file.stem}_report.html"
    )

    report.show_html(
        filepath=str(output_file),
        open_browser=False
    )

    print(
        f"Saved: {output_file.name}"
    )

print(
    "\nAll reports generated."
)

Generating report for day_01.parquet...


                                             |          | [  0%]   00:00 -> (? left)

Report c:\Users\ali.farhat\Documents\simbox-project\reports\day_01_report.html was generated.
Saved: day_01_report.html
Generating report for day_02.parquet...


                                             |          | [  0%]   00:00 -> (? left)

Report c:\Users\ali.farhat\Documents\simbox-project\reports\day_02_report.html was generated.
Saved: day_02_report.html
Generating report for day_03.parquet...


...

Report c:\Users\ali.farhat\Documents\simbox-project\reports\day_13_report.html was generated.
Saved: day_13_report.html
Generating report for day_14.parquet...


                                             |          | [  0%]   00:00 -> (? left)

Report c:\Users\ali.farhat\Documents\simbox-project\reports\day_14_report.html was generated.
Saved: day_14_report.html
Generating report for day_15.parquet...


                                             |          | [  0%]   00:00 -> (? left)

Report c:\Users\ali.farhat\Documents\simbox-project\reports\day_15_report.html was generated.
Saved: day_15_report.html

All reports generated.


### Summary table

In [12]:
summary_rows = []

for profile_file in profile_files:

    df = pd.read_parquet(
        profile_file
    )

    summary_rows.append({
        "profile_file": profile_file.name,
        "rows": len(df),
        "fraud_profiles": (
            df["label"] == 1
        ).sum(),
        "normal_profiles": (
            df["label"] == 0
        ).sum()
    })

summary_df = pd.DataFrame(
    summary_rows
)

display(summary_df)

,profile_file,rows,fraud_profiles,normal_profiles
0,day_01.parquet,28004,1540,26464
1,day_02.parquet,28401,1646,26755
2,day_03.parquet,28501,1662,26839
3,day_04.parquet,28719,1658,27061
4,day_05.parquet,28648,1639,27009
5,day_06.parquet,28468,1563,26905
6,day_07.parquet,28916,1674,27242
7,day_08.parquet,28771,1676,27095
8,day_09.parquet,28761,1691,27070
9,day_10.parquet,28535,1671,26864


---
### Generate and save report for all profiles

In [14]:
all_profiles_df = pd.read_parquet(PROFILE_PATH / "all_profiles.parquet")

report = sv.analyze(
    all_profiles_df,
    target_feat= "label",
    pairwise_analysis="on"
)

report_path = (REPORT_PATH / "all_profiles_report.html")

report.show_html(
    filepath=str(report_path),
    open_browser=False
)

print(f"Report saved to:\n{report_path}")

                                             |          | [  0%]   00:00 -> (? left)

Report c:\Users\ali.farhat\Documents\simbox-project\reports\all_profiles_report.html was generated.
Report saved to:
c:\Users\ali.farhat\Documents\simbox-project\reports\all_profiles_report.html
